# sandbox

> Running Python fences produced by a model.

`PyFenceCallback` runs fenced Python in a sandboxed namespace and returns the output to the model. This gives code execution to models that cannot emit reliable tool calls.

The module also contains utilities for consuming asynchronous streams from synchronous code.

::: {.callout-note}
The sandbox can move to `safepyrun`. The async utilities can move beside the streaming backends.
:::

In [ ]:
#| default_exp sandbox

In [ ]:
#| export
import asyncio, io, re, subprocess, warnings
from contextlib import contextmanager, redirect_stdout
from concurrent.futures import ThreadPoolExecutor
from fastcore.all import L, store_attr, patch
from safepyrun import RunPython
from urai.core import ChatCallback, resp_text
from urai.chat import Chat
from urai.eval import matches_

In [ ]:
#| hide
import threading, time
from fastcore.test import test_eq, test_fail
from urai.core import Resp
from urai.opts import RUNTIMES, ChatOpts, Runtime, register_runtime

## Fences

The *last* fence, not the first, because a model that revises itself leaves both in the reply and means the second one.

In [ ]:
#| export
_pyfence_re = re.compile(r'^```(?:python|py)[ \t]*\n(.*?)\n```', re.DOTALL | re.MULTILINE)

def extract_code(text):
    "Code of the last ```python fence in `text`, else None."
    return ms[-1] if (ms := _pyfence_re.findall(text or '')) else None

def mk_result_fence(out):
    "Feed a code result back, asking for prose or for more code."
    return (f'```result\n{out}\n```\n\nIf this answers the request, reply with the final answer in '
            'prose; only write another ```python block if you need to run more code.')

In [ ]:
test_eq(extract_code('```python\nprint(1)\n```'), 'print(1)')
test_eq(extract_code('```py\nx = 2\n```'), 'x = 2')             # both spellings
test_eq(extract_code('first\n```python\na\n```\nno, wait\n```python\nb\n```'), 'b')
test_eq(extract_code('no code here'), None)
assert '```result\n7\n```' in mk_result_fence(7)

## Running it

The namespace persists across rounds of one chat, so a model can define a function in one fence and call it in the next. An exception comes back as a string for the same reason a failing tool does: the model can read it and try again.

In [ ]:
#| export
@patch
def run_py(self:Chat, code, ban_defs=False, g=None):
    "Run `code` in this chat's sandboxed, persistent namespace. Returns stdout plus the last expression."
    if not hasattr(self, '_pyrun'): self._pyrun = RunPython(g=g, ban_defs=ban_defs)
    buf = io.StringIO()
    try:
        with redirect_stdout(buf): res = run_coro(self._pyrun(code))
        out = buf.getvalue()
        if res is not None: out = (out + '\n' if out else '') + repr(res)
        return out or '(ok)'
    except Exception as e: return f'{type(e).__name__}: {e}'

def run_coro(coro):
    "Run an awaitable to completion from sync code, even inside a running event loop."
    try: asyncio.get_running_loop()
    except RuntimeError: return asyncio.run(coro)
    with ThreadPoolExecutor(1) as ex: return ex.submit(asyncio.run, coro).result()

In [ ]:
class _FenceChat(Chat):
    "A chat whose replies are scripted, for testing the fence loop without a model."
    _runtime, ctx_limit, token_count = 'fence', 8192, 0
    def __init__(self, model=None, *, script=(), **kw):
        self.script, self.steps = list(script), 0
        self._setup(model, ChatOpts.create(kw.pop('opts', None), **kw))
    def _send(self, msg, **kw):
        self.turn_msg = self.mk_msg(msg)
        if self.turn_msg: self.hist.append(self.turn_msg)
        text = self.script[min(self.steps, len(self.script) - 1)] if self.script else ''
        self.steps += 1
        self.turn_res = Resp({'role': 'assistant', 'content': text})
        self.hist.append(dict(self.turn_res))
        from urai.core import run_cbs
        for _ in run_cbs(self, 'after_response'): pass
        return self.turn_res

register_runtime(Runtime('fence', _FenceChat, ('fence-',)))
c = _FenceChat()

In [ ]:
test_eq(c.run_py('1 + 1'), '2')
test_eq(c.run_py('print("hi")'), 'hi\n')
test_eq(c.run_py('x = 5'), '(ok)')          # a statement produces nothing to show
test_eq(c.run_py('x * 2'), '10')            # ...but the namespace persisted

In [ ]:
test_eq(c.run_py('1/0'), 'ZeroDivisionError: division by zero')
test_eq(c.run_py('import math\nmath.sqrt(16)'), '4.0')

In [ ]:
# a function defined in one round can be called in the next
test_eq(c.run_py('def double(n): return n * 2'), '(ok)')
test_eq(c.run_py('double(21)'), '42')
test_eq(run_coro(asyncio.sleep(0, 'done')), 'done')

## The loop

`done` decides when to stop feeding results back; without one, the loop ends when the model writes no more fences. `max_rounds` is the safety cap on top of that, for a model that keeps writing code forever.

Two policies come ready: stop when the last output contains what you were looking for, or ask the model itself whether the request is now answered.

In [ ]:
#| export
def task_complete(chat):
    "`done` policy: ask the model, in isolation, whether the latest result completes the request."
    convo = '\n'.join(f"{m.get('role','?')}: {resp_text(m)}" for m in chat.hist[-6:])
    return chat.classify(convo, ['complete', 'needs_more_work']) == 'complete'

def output_matches(expected):
    "`done` policy: stop once the last code output contains `str(expected)`."
    return lambda chat: matches_(getattr(chat, 'turn_code_out', ''), expected)

class PyFenceCallback(ChatCallback):
    "Run ```python fences, feed results back, and loop until `done(chat)` or no fence is left."
    order = 50
    def __init__(self,
                 max_rounds=5,       # safety cap on rounds of code in one turn
                 done=None,          # `done(chat) -> bool`; None means "stop when there is no fence"
                 pyrun:callable=None # runner to use instead of `chat.run_py`
                ):
        store_attr(); self._depth = 0; self._warned = False

    def _active(self):
        "Only the last-registered callback drives a turn, so a per-call `done` beats a persistent one."
        cbs = getattr(self.chat, 'cbs', L())
        later = self in cbs and any(isinstance(cb, PyFenceCallback) for cb in cbs[cbs.index(self)+1:])
        if later and not self._warned:
            warnings.warn('This chat already has a PyFenceCallback; the last-registered one drives the turn.')
            self._warned = True
        return not later

    def _run_code(self):
        "Run the last fence. `(output, done)`, or `(None, False)` at the cap or with no fence."
        if self._depth >= self.max_rounds: return None, False
        code = extract_code(resp_text(self.chat.turn_res))
        if not code: return None, False
        tc = {'function': {'name': 'python', 'arguments': {'code': code}}}
        fn = self.pyrun or self.chat.run_py
        out = 'Denied by human' if (self.chat.approve and not self.chat.approve(tc)) else fn(code)
        self.chat.turn_code_out = out
        return out, bool(self.done and self.done(self.chat))

    def after_response(self):
        "Feed the result back: synchronously, or as a chunk generator during a streamed turn."
        if not self._active(): return
        if getattr(self.chat, '_streaming', False): return self._after_stream()
        out, done = self._run_code()
        if out is None: return
        self._depth += 1
        try:
            # when done, record the clean output rather than spending another model call on it
            if done: self.chat.hist.append(self.chat.fmt2hist([out])[0])
            else: self.chat._send(mk_result_fence(out))
        finally: self._depth -= 1

    def _after_stream(self):
        "Emit the result into the stream, then stream the follow-up reply unless we are done."
        out, done = self._run_code()
        if out is None: return
        self._depth += 1
        try:
            if done:
                self.chat.hist.append(self.chat.fmt2hist([out])[0])
                yield f'\n\n```result\n{out}\n```\n'
                return
            rf = mk_result_fence(out)
            yield '\n\n' + rf + '\n\n'
            yield from self.chat._stream(rf)
        finally: self._depth -= 1

In [ ]:
c = _FenceChat(script=['```python\n6 * 7\n```', 'The answer is 42.'],
               cbs=[PyFenceCallback()])
c('what is 6 times 7?')
test_eq(resp_text(c.turn_res), 'The answer is 42.')
test_eq(c.turn_code_out, '42')
assert '```result\n42\n```' in c.hist[2]['content']    # the result was fed back

In [ ]:
# with no fence in the reply there is nothing to run, and no extra model call
c = _FenceChat(script=['Just prose.'], cbs=[PyFenceCallback()])
c('hello')
test_eq(c.steps, 1)
test_eq(len(c.hist), 2)

In [ ]:
# `done` stops the loop and records the clean output, without asking the model again
c = _FenceChat(script=['```python\n6 * 7\n```'], cbs=[PyFenceCallback(done=output_matches(42))])
c('what is 6 times 7?')
test_eq(c.steps, 1)
test_eq(c.hist[-1]['content'], '42')

In [ ]:
c = _FenceChat(script=['```python\n1\n```'], cbs=[PyFenceCallback(max_rounds=2)])
c('go')
test_eq(c.steps, 3)          # the first reply, then two rounds of code, then the cap
test_eq(c.turn_code_out, '1')

In [ ]:
# approval covers a fence exactly as it covers a tool call
c = _FenceChat(script=['```python\n1/0\n```', 'ok'], cbs=[PyFenceCallback()],
               approve=lambda tc: False)
c('go')
test_eq(c.turn_code_out, 'Denied by human')

In [ ]:
# a caller's own runner replaces `chat.run_py` entirely
seen = []
c = _FenceChat(script=['```python\nwhatever\n```', 'ok'],
               cbs=[PyFenceCallback(pyrun=lambda code: seen.append(code) or 'from my runner')])
c('go')
test_eq((seen, c.turn_code_out), (['whatever'], 'from my runner'))

In [ ]:
# two registered callbacks: the last one drives, and says so once
c = _FenceChat(script=['```python\n1\n```', 'ok'])
first_cb, last_cb = PyFenceCallback(), PyFenceCallback(done=output_matches(1))
c.add_cbs([first_cb, last_cb])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    c('go')
test_eq(len(caught), 1)
assert 'last-registered' in str(caught[0].message)

## Driving async code from sync code

`sync_iter` runs an async generator on one event loop in one background thread. Repeated `asyncio.run` calls would move a streaming response between event loops and break the connection.

Abandoning the returned generator stops only its consumer. The producer continues until the response ends. Setting `stop` closes the source, then drains items already queued.

In [ ]:
#| export
def sync_iter(agen_fn, stop=None):
    "Drive the async generator from `agen_fn()` in sync code. Set `stop` to end the source stream."
    from queue import Queue
    from threading import Thread
    q, done = Queue(), object()
    async def _pump():
        agen = agen_fn()
        try:
            async for o in agen:
                q.put(o)
                if stop is not None and stop.is_set(): break
        except BaseException as e: q.put(e)
        finally:
            try: await agen.aclose()
            except BaseException: pass
            q.put(done)
    t = Thread(target=lambda: asyncio.run(_pump()), daemon=True)
    t.start()
    try:
        while (o := q.get()) is not done:
            if isinstance(o, BaseException): raise o
            yield o
    finally: t.join(timeout=5)

@contextmanager
def killed_on_exit(proc, timeout=2):
    """Wait for `proc` on success. Terminate it if the block exits early or raises."""
    try: yield proc
    except BaseException:
        if proc.poll() is None:
            proc.terminate()
            try: proc.wait(timeout)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
        for p in (proc.stdin, proc.stdout, proc.stderr):
            if p is not None:
                try: p.close()
                except Exception: pass
        raise
    else: proc.wait()

In [ ]:
async def _count(n=3):
    for i in range(n):
        await asyncio.sleep(0)
        yield i

test_eq(list(sync_iter(_count)), [0, 1, 2])

In [ ]:
async def _boom():
    yield 1
    raise ValueError('from the source')

it = sync_iter(_boom)
test_eq(next(it), 1)
test_fail(lambda: next(it), contains='from the source')   # the source's error, not a wrapper's

In [ ]:
# `stop` ends the source, not just the consumer
closed = []
async def _forever():
    try:
        i = 0
        while True:
            await asyncio.sleep(0.01); yield i; i += 1
    finally: closed.append(True)

stop = threading.Event()
out = []
for o in sync_iter(_forever, stop):
    out.append(o)
    if len(out) == 3: stop.set()
test_eq(out[:3], [0, 1, 2])
test_eq(closed, [True])      # the source generator was closed, not left running
assert len(out) <= 5         # a chunk or two may already be in flight, but no more

In [ ]:
p = subprocess.Popen(['sleep', '30'])
try:
    with killed_on_exit(p): raise KeyboardInterrupt
except KeyboardInterrupt: pass
test_eq(p.poll() is not None, True)          # killed, not waited for

In [ ]:
p = subprocess.Popen(['true'])
with killed_on_exit(p): pass
test_eq(p.returncode, 0)                     # a normal exit still waits

In [ ]:
#| hide
del RUNTIMES['fence']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()